In [8]:
%pip install sentence-transformers langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. 配置設定
# 請確認這路徑正確指向你的 PDF
PDF_PATH = "data/2306.08045v2.pdf" 
DB_PATH = "faiss_index"

# 2. 載入本地模型 (不需聯網，直接在你的筆電上運算)
print("🔄 正在啟動本地 Embedding 引擎...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. 讀取並分塊 (解決 pypdf 錯誤)
if not os.path.exists(PDF_PATH):
    print(f"❌ 錯誤：找不到檔案 {PDF_PATH}，請確認路徑。")
else:
    print("🏗️ 正在讀取並處理文件...")
    loader = PyPDFLoader(PDF_PATH)
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = splitter.split_documents(docs)

    # 4. 建立向量庫並持久化 (解決開發階段反覆運算的麻煩)
    if os.path.exists(DB_PATH):
        print("📂 檢測到已存在的向量庫，直接載入。")
        vectorstore = FAISS.load_local(DB_PATH, embeddings, allow_dangerous_deserialization=True)
    else:
        print("⚡ 建立向量庫中...")
        vectorstore = FAISS.from_documents(splits, embeddings)
        vectorstore.save_local(DB_PATH)
        print("✅ 向量庫已保存到硬碟。")

    # 5. 測試檢索
    query = "這篇論文的核心貢獻是什麼？"
    results = vectorstore.similarity_search(query, k=2)
    print("\n🔍 檢索結果：", results[0].page_content[:200])

🔄 正在啟動本地 Embedding 引擎...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 703.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


❌ 錯誤：找不到檔案 data/your_paper.pdf，請確認路徑。


In [6]:
import os
# 1. 查看當前工作目錄
print(f"目前工作目錄: {os.getcwd()}")
# 2. 列出當前目錄下的所有檔案與資料夾
print(f"目錄下的檔案: {os.listdir()}")

目前工作目錄: d:\AI\RAG_project
目錄下的檔案: ['.env', '.gitignore', '.venv', 'data', 'Images_RAG.ipynb', 'index', 'main.py', 'requirements.txt', 'Text_RAG - 複製.ipynb', 'Text_RAG.ipynb', 'venv']
